# 04 · Read the analysis from left to right

## Context

CubeDynamics makes the order of operations visible: a cube flows through small
verbs instead of disappearing inside a long function call.

## Question

Does the pipe grammar change the calculation, or only make the method easier
to read and extend?

## Analysis story

We compare direct and piped standardization exactly, then compose a regional
anomaly in one compact expression.


### Data used in this lesson

Every value comes from the PRISM Group at Oregon State University's AN91d
daily 4 km climate product. This repository carries a small Boulder-region
extract for 1–30 January 2024 so the lesson runs offline without replacing
observations with generated values. The [data validation page](../validation/data.md)
records source URLs, terms, checksums, bounds, units, and acceptance tests.

## Prepare · Select an observed temperature cube

In [ ]:
from pathlib import Path

import xarray as xr

# Find the repository from either a root-level documentation build or a kernel
# started beside this notebook, then open the checksum-controlled PRISM extract.
data_path = next(
    candidate / "data" / "vignettes" / "prism_boulder_january_2024.nc"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "data" / "vignettes" / "prism_boulder_january_2024.nc").exists()
)
prism = xr.open_dataset(data_path, engine="scipy").load()

# These assertions are part of the teaching contract: official source,
# canonical cube dimensions, complete daily time, and declared Celsius units.
assert prism.attrs["source"] == "PRISM Group, Oregon State University"
assert prism.attrs["is_synthetic"] == 0
assert prism.sizes == {"time": 30, "y": 24, "x": 24}
assert prism["tmax"].attrs["units"] == "degC"

# Keep the labeled PRISM DataArray intact as it enters the grammar.
cube = prism["tmax"]
cube

## Pipe · Verify equivalence, then compose

In [ ]:
import numpy as np
from cubedynamics import pipe, verbs as v

# The grammar must preserve the mathematical definition of a z-score.
direct = (cube - cube.mean("time")) / cube.std("time")
through_grammar = (pipe(cube) | v.zscore(dim="time")).unwrap()
np.testing.assert_allclose(through_grammar, direct, rtol=1e-6, atol=1e-6)

# The method reads left to right: anomaly first, then spatial mean.
regional_anomaly = (
    pipe(cube)
    | v.anomaly(dim="time")
    | v.mean(dim=("y", "x"))
).unwrap()

## Figure · See the effect of each method

In [ ]:
import matplotlib.pyplot as plt

site = cube.isel(y=12, x=12)
fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True, constrained_layout=True)
site.plot(ax=axes[0], marker="o", color="#8f513b")
axes[0].set_title("Observed PRISM maximum temperature")
axes[0].set_ylabel("°C")
through_grammar.isel(y=12, x=12).plot(ax=axes[1], marker="o", color="#3b6d74")
axes[1].axhline(0, color="0.4", linewidth=0.8)
axes[1].set_title("pipe(cube) | v.zscore(dim='time')")
axes[1].set_ylabel("Standard deviations")
regional_anomaly.plot(ax=axes[2], marker="o", color="#5b6848")
axes[2].axhline(0, color="0.4", linewidth=0.8)
axes[2].set_title("pipe(cube) | v.anomaly(...) | v.mean(dim=('y', 'x'))")
axes[2].set_ylabel("Regional anomaly (°C)")
plt.show()

## What the figure tells us

The exact comparison proves that the pipe is a composition language, not a new
statistical definition. Its advantage is a method that remains visible.

## Try the next variation

Replace the final mean with a variance and explain how the question changes.